# 07 – Thử bước kiểm chứng

Dựng tay một gói căn cứ nhỏ cùng 4 đề xuất (đúng, mã đoạn lạ, câu trích bịa, con số bịa) và chạy `verify_all`.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from app.core.types import Chunk
from app.llm.schema import GoiCanCu, KetQuaTongHop, NhanDinhDeXuat, TrichDanNguon
from app.verify.pipeline import verify_all

chunk = Chunk(
    doan_trich_id="DT-MERO-01",
    tai_lieu_id="TL-MERO",
    muc_cha_id=None,
    nhom_nguon="nhan_thuoc",
    nguon="openfda",
    tang_tin_cay=3,
    hoat_chat_ids=["meropenem"],
    muc="lieu_than",
    duong_dan_muc="Dosage and Administration > Renal Impairment",
    loai_noi_dung="van_ban",
    ngon_ngu="en",
    phien_ban_nguon="2025-01-01",
    url_goc=None,
    noi_dung_goc=(
        "For creatinine clearance 10 to 25 mL/min, administer one-half the "
        "recommended dose every 12 hours."
    ),
    noi_dung_nhung="[Meropenem | FDA] For creatinine clearance 10 to 25 mL/min...",
)

goi = GoiCanCu(
    ca_id="TEST-01",
    van_ban_ca="[XN-01] Creatinin 212 umol/L\n[CT-01] Meropenem 1 g",
    ban_ghi_ids=["XN-01", "CT-01"],
    dong_y_lenh_ids=["CT-01"],
    so_cho_phep=[212.0, 16.0, 1.0],
    doan=[chunk],
)

In [ ]:
de_xuat_dung = NhanDinhDeXuat(
    chi_tiet_y_lenh_id=["CT-01"],
    yeu_to=6,
    ket_luan="can_xem",
    muc_do="nghiem_trong",
    nhan_xet="CrCl 16,0 mL/phút: cần xem xét chỉnh liều mỗi 12 giờ.",
    ban_ghi_ids=["XN-01", "CT-01"],
    trich_dan=[
        TrichDanNguon(
            doan_trich_id="DT-MERO-01",
            trich_nguyen_van="administer one-half the recommended dose every 12 hours",
        )
    ],
)

de_xuat_ma_doan_la = de_xuat_dung.model_copy(
    update={
        "trich_dan": [
            TrichDanNguon(
                doan_trich_id="DT-KHONG-CO",
                trich_nguyen_van="administer one-half the recommended dose every 12 hours",
            )
        ]
    }
)

de_xuat_trich_bia = de_xuat_dung.model_copy(
    update={
        "trich_dan": [
            TrichDanNguon(
                doan_trich_id="DT-MERO-01",
                trich_nguyen_van="administer full recommended dose every 8 hours",
            )
        ]
    }
)

de_xuat_so_bia = de_xuat_dung.model_copy(
    update={"nhan_xet": "CrCl 28 mL/phút: cần xem xét chỉnh liều mỗi 12 giờ."}
)

tong_hop = KetQuaTongHop(
    nhan_dinh=[de_xuat_dung, de_xuat_ma_doan_la, de_xuat_trich_bia, de_xuat_so_bia]
)

In [ ]:
ket_qua = verify_all(tong_hop, goi)
ten_ca = ["1. Đúng", "2. Mã đoạn lạ", "3. Câu trích bịa", "4. Con số bịa"]
for nhan, item in zip(ten_ca, ket_qua, strict=True):
    print(f"{nhan}: dat={item.dat}, loi={item.loi}")